# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
import tensorflow as tf
import tensorflow_io as tfio # Crucial for .ogg files on Kaggle
import pandas as pd
import numpy as np
import os
import librosa
import IPython.display as ipd
import matplotlib.pyplot as plt
from tqdm import tqdm
import gc
import cv2

2026-03-30 16:53:59.988474: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774889640.171628      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774889640.221594      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774889640.658330      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774889640.658367      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774889640.658370      24 computation_placer.cc:177] computation placer alr

In [2]:
BASE_DIR = "/kaggle/input/competitions/birdclef-2026" #os is used to connect stuff from across our file space since
# not all files are in the same folder
AUDIO_DIR = os.path.join(BASE_DIR, "train_audio/") #This points to the specific folder INSIDE that contains the .ogg files

In [3]:
df = pd.read_csv(os.path.join(BASE_DIR, "train.csv"))
df['full_path'] = df.apply(
    lambda row: os.path.join(AUDIO_DIR, str(row['filename']).strip()), 
    axis=1
)
if os.path.exists(df['full_path'].iloc[0]):
    print("✅ Path logic is correct! File found.")
    print(f"Sample path: {df['full_path'].iloc[0]}") #this to check if file is found or not
else:
    print("❌ File not found. Check if 'train_audio' is unzipped correctly.")

✅ Path logic is correct! File found.
Sample path: /kaggle/input/competitions/birdclef-2026/train_audio/1161364/iNat1216197.ogg


In [4]:
from sklearn.model_selection import train_test_split
bird_counts=df['primary_label'].value_counts()
keep_birds = bird_counts[bird_counts >= 2].index
df_filtered= df[df['primary_label'].isin(keep_birds)].reset_index(drop=True)
unique_labels = sorted(df_filtered['primary_label'].unique()) #takes unique labels and sorts in order
label_to_id = {label: i for i, label in enumerate(unique_labels)} #creates a dictionary which relates label wrt 0,1,2,3,.. id
df_filtered['label_id'] = df_filtered['primary_label'].map(label_to_id) #makes a new column with the mapped id
train_df, val_df = train_test_split(
    df_filtered, 
    test_size=0.2, 
    random_state=42, #seed
    stratify=df_filtered['primary_label'],
    shuffle=True
#is a parameter used during data splitting (usually in train_test_split) to ensure that the
    #proportion of classes in your training and validation sets remains the same as in your original dataset.
)

In [5]:
num_mel=128
num_of_bird_species= keep_birds.shape[0]
print(num_of_bird_species)
hop_length=512
width= (5*32000)/hop_length #its 312.5  but we will use 312

202


In [6]:
def process_audio_efficientB0(file_path,label):
    path_string = file_path.numpy().decode('utf-8') #we need as string, its currently as object tensor. and librosa uses utf-8, so we decode
    #in that
    y, sr = librosa.load(file_path.numpy(), sr=32000, duration=5.0) #used to load at 5 seconds duration at 32KHz
    target_length = 5 * 32000  # since audio looks at frames, the number of samples is duration*frequency
    if len(y) < target_length: #all data is not the duration, so we need to pad it
        y=np.pad(y,(0,target_length - len(y)))
    spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128, fmin=20, fmax=16000) 
    spec = spec[:, :312]
    #we use between min 20Hz to get rid of noise due to earth
    #and frequency max is apparently due to Nyquist-Shannon Theorem fmax is half of sampled rate of 32Khz
    #one thing to note is that melspectrogram uses output in terms of power instead of amplitude, so using amplitude_to_db actually 
    # reduces db by a factor of 2, so we use power_to_db instead
    spec_db = librosa.power_to_db(spec, ref=np.max)
    spec_db= (spec_db-spec_db.min())/(spec_db.max()-spec_db.min()+1e-6) #normalization step for audio stuff, 1e-6 is there to prevent 
    #zero division
    spec_db=spec_db[...,np.newaxis] #makes from 2D to 3D
    spec_db = np.repeat(spec_db, 3, axis=-1) #because efficientnet requires rgb channel
    return spec_db.astype(np.float32),np.int32(label)

In [7]:
def process_audio_efficientB3(file_path,label):
    path_string = file_path.numpy().decode('utf-8') #we need as string, its currently as object tensor. and librosa uses utf-8, so we decode
    #in that
    y, sr = librosa.load(file_path.numpy(), sr=32000, duration=5.0) #used to load at 5 seconds duration at 32KHz
    target_length = 5 * 32000  # since audio looks at frames, the number of samples is duration*frequency
    if len(y) < target_length: #all data is not the duration, so we need to pad it
        y=np.pad(y,(0,target_length - len(y)))
    spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=256, fmin=20, fmax=16000) 
    spec = spec[:, :256]
    #we use between min 20Hz to get rid of noise due to earth
    #and frequency max is apparently due to Nyquist-Shannon Theorem fmax is half of sampled rate of 32Khz
    #one thing to note is that melspectrogram uses output in terms of power instead of amplitude, so using amplitude_to_db actually 
    # reduces db by a factor of 2, so we use power_to_db instead
    spec_db = librosa.power_to_db(spec, ref=np.max)
    spec_db= (spec_db-spec_db.min())/(spec_db.max()-spec_db.min()+1e-6) #normalization step for audio stuff, 1e-6 is there to prevent 
    #zero division
    # If Librosa gives us 256x251 (slight variation), we force it to 256x256
    if spec_db.shape != (256, 256):
        spec_db = cv2.resize(spec_db, (256, 256))
    spec_db=spec_db[...,np.newaxis] #makes from 2D to 3D
    spec_db = np.repeat(spec_db, 3, axis=-1) #because efficientnet requires rgb channel
    return spec_db.astype(np.float32),np.int32(label)

In [8]:
#for B0
def prepare_dataset(df,batch_size=16,shuffle=False):
    file_paths = df['full_path'].values
    labels = df['label_id'].values
    ds = tf.data.Dataset.from_tensor_slices((file_paths, labels)) #creates a dataset of the file path and labe;
    if shuffle: #to switch shuffle on and off
        ds = ds.shuffle(buffer_size=len(df))
    ds = ds.map(
        lambda x, y: tf.py_function(process_audio_efficientB0, [x, y], [tf.float32, tf.int32]),
        num_parallel_calls=tf.data.AUTOTUNE
    )
    """
        So there is quite abit to explain here,
        1)first is the mapping function, its just a transformation tool.
        2)since tensorflow and librosa are not exactly compatible(due to the nature of how they run), we use a external function called
        py_function, it acts like the middle man allowing process_audio to run in python.
        3)processes multiple audio files at the same time, essentially a speed boost, tensorflow looks at the number of cores the device has
    """
    def set_shape(img,label):
        img.set_shape((num_mel,312,3)) #when we move to python in py_function, it tends to forget the shape
        label.set_shape([]) #[] indicates its a scalar
        return img,label
    ds = ds.cache() # Stores the processed spectrograms in memory
    ds = ds.map(set_shape,num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size,drop_remainder=True).prefetch(tf.data.AUTOTUNE) #drop_remainder=True is good for 
    #training to keep batch sizes consistent
    # data goes from (128, 312, 3) to (batch_size, 128, 312, 3), this is alot faster as we are processing 16 at a time
    #prefetch is another function we use for faster performance, prevents idling of cpu/gpu
    return ds


In [9]:
def _bytes_feature(value):
    """Returns a bytes_list from a string / byte."""
    if isinstance(value, type(tf.constant(0))):
        value = value.numpy() # Get value from a tensor if needed
    return tf.train.Feature(bytes_list=tf.train.BytesList(value=[value]))

def _float_feature(value):
    """Returns a float_list from a float / double."""
    return tf.train.Feature(float_list=tf.train.FloatList(value=[value]))

def _int64_feature(value):
    """Returns an int64_list from a bool / enum / int / uint."""
    return tf.train.Feature(int64_list=tf.train.Int64List(value=[value]))

In [10]:
def create_tfrecords(df, folder_name, num_shards=10):
    os.makedirs(folder_name, exist_ok=True)
    samples_per_shard = len(df) // num_shards
    
    for shard in range(num_shards):
        shard_path = os.path.join(folder_name, f"shard_{shard}.tfrecord")
        with tf.io.TFRecordWriter(shard_path) as writer:
            start, end = shard * samples_per_shard, (shard + 1) * samples_per_shard
            if shard == num_shards - 1: end = len(df)
            
            for i in tqdm(range(start, end), desc=f"Shard {shard}"):
                row = df.iloc[i]
                try:
                    # Convert to Tensors as verified in our diagnostic
                    p_tensor = tf.constant(row['full_path'])
                    l_tensor = tf.constant(row['label_id'])
                    
                    # Pass Tensors to your working B3 function
                    image, label = process_audio_efficientB3(p_tensor, l_tensor)
                    
                    # Serialize the result
                    example = tf.train.Example(features=tf.train.Features(feature={
                        'image': _bytes_feature(image.astype(np.float16).tobytes()),
                        'label': _int64_feature(int(label))
                    }))
                    writer.write(example.SerializeToString())
                except Exception as e:
                    # If ONE file fails, we want to know why without stopping the whole shard
                    if i == start: print(f"Shard {shard} error on first file: {e}")
                    continue 
        # Crucial for Kaggle RAM limits
        gc.collect()

# Run the creation process
create_tfrecords(train_df, "/kaggle/working/train_tfrecords", num_shards=15)
create_tfrecords(val_df, "/kaggle/working/val_tfrecords", num_shards=3)

Shard 2: 100%|██████████| 2371/2371 [01:27<00:00, 27.20it/s]


In [11]:
IMG_SIZE = 256 # Updated for B3 optimal scaling

def parse_tfrecord(example_proto):
    feature_description = {
        'image': tf.io.FixedLenFeature([], tf.string),
        'label': tf.io.FixedLenFeature([], tf.int64),
    }
    example = tf.io.parse_single_example(example_proto, feature_description)
    
    # Decode and Reshape
    image = tf.io.decode_raw(example['image'], tf.float16)
    image = tf.cast(image, tf.float32)
    image = tf.reshape(image, (IMG_SIZE, IMG_SIZE, 3))
    image = image * 255.0
    image = tf.clip_by_value(image, 0.0, 255.0)
    label = tf.cast(example['label'], tf.int32)
    
    return image, label

def get_dataset(file_pattern, batch_size=16, shuffle=False):
    files = tf.data.Dataset.list_files(file_pattern)
    ds = files.interleave(tf.data.TFRecordDataset, cycle_length=tf.data.AUTOTUNE, num_parallel_calls=tf.data.AUTOTUNE)
    
    if shuffle:
        ds = ds.shuffle(2048)
    
    ds = ds.map(parse_tfrecord, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

num_mel=128
eff_B0=tf.keras.applications.EfficientNetB0(
    include_top=False,
    weights='imagenet',
    input_shape=(num_mel,312,3),
    pooling='avg',
)
eff_B0.trainable = False
model_B0= tf.keras.Sequential([
    eff_B0,
    tf.keras.layers.BatchNormalization(), # Helps stabilize training
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dropout(0.3),          # Prevents overfitting on the small dataset
    tf.keras.layers.Dense(num_of_bird_species, activation='softmax')
])
model_B0.summary()
#model setup, we use SparseCategoricalCrossentropy in the event the the labels are integers like 0,1,2,3,...
#and apparently Adam is used frequently
#we are using metric as accuracy, but we should care about F1 score and validation loss(means new data it has never seen before)
#If you see your training loss jumping up and down wildly in the first 2 epochs, it means 1e-3 is too high for your specific data—try 1e-4
model_B0.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)
"""
Callbacks/safety nets(mostly to save time):
EarlyStopping: Stops the training if the validation loss doesn't improve for 5-10 epochs.

ModelCheckpoint: Automatically saves your "best" model weights to a file so you don't lose progress if your laptop sleeps or crashes.

ReduceLROnPlateau: If the model's progress slows down, this "shrinks" the learning rate to help the model find the very bottom of the 
loss curve.
"""
callbacks_list = [
        tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),#prevents overfitting, patience is if in a set number of
        # epochs theres no improvement in validation loss, then it will stop to prevent it getting worse.
        tf.keras.callbacks.ModelCheckpoint("best_bird_model.keras", save_best_only=True), #ensures you don't lose
        #your progress if your Mac crashes or the battery dies.
        tf.keras.callbacks.ReduceLROnPlateau(factor=0.2, patience=3), #As your model gets closer to the "answer," a large Learning Rate (LR)
        #might make it overshoot the goal. This callback shifts the model into a lower gear.
        tf.keras.callbacks.CSVLogger('/kaggle/working/training_log_B0.csv', append=True),
]
train_ds = prepare_dataset(train_df, shuffle=True).cache('/kaggle/working/train_b0_cache')
val_ds = prepare_dataset(val_df, shuffle=False).cache('/kaggle/working/val_b0_cache')
history_warmup_B0 = model_B0.fit(train_ds,
                           validation_data=val_ds,
                           epochs=5,
                           callbacks= callbacks_list,
)
eff_B0.trainable = True
model_B0.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5), 
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)
EPOCHS = 30 # Start with 30; EarlyStopping will likely cut it short

history_tuning_B0= model_B0.fit(
    train_ds,
    validation_data=val_ds, #this also creates val_accuracy data
    epochs=EPOCHS,
    callbacks=callbacks_list
)
#history is kinda like a black box or just all info, note it has history.history attritbute which we use to access everything.

In [12]:
img_size=256 #efficientNetB3, expects to run better in this
eff_B3=tf.keras.applications.EfficientNetB3(
    include_top=False,
    weights='imagenet',
    input_shape=(img_size,img_size,3),
    pooling='max', #max picksup sharp features
)
eff_B3.trainable = True
model_B3= tf.keras.Sequential([
    eff_B3,
    tf.keras.layers.BatchNormalization(), # Helps stabilize training
    tf.keras.layers.Dense(512, activation='swish'), #efficientnet uses swish so its more familiar with it
    tf.keras.layers.Dropout(0.5),     # Prevents overfitting on the small dataset
    tf.keras.layers.Dense(256, activation='swish'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(num_of_bird_species, activation='softmax')
])
model_B3.summary()
model_B3.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(), #label_smoothing prevents overfitting
    metrics=['accuracy'],
)
callbacks_list = [
        tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),#prevents overfitting, patience is if in a set number of
        # epochs theres no improvement in validation loss, then it will stop to prevent it getting worse.
        tf.keras.callbacks.ModelCheckpoint("best_bird_model.keras", save_best_only=True), #ensures you don't lose
        #your progress if your Mac crashes or the battery dies.
        tf.keras.callbacks.ReduceLROnPlateau(factor=0.2, patience=3), #As your model gets closer to the "answer," a large Learning Rate (LR)
        #might make it overshoot the goal. This callback shifts the model into a lower gear.
        tf.keras.callbacks.CSVLogger('/kaggle/working/training_log_B3.csv', append=True),
]
train_ds = get_dataset("/kaggle/working/train_tfrecords/*.tfrecord", batch_size=16, shuffle=True)
val_ds = get_dataset("/kaggle/working/val_tfrecords/*.tfrecord", batch_size=16)
"""
* acts as a "match-all" symbol. It tells TensorFlow: "Don't just look for one file; look for every single file in this folder 
that ends with .tfrecord."
"""
EPOCHS = 30 # Start with 30; EarlyStopping will likely cut it short

history_tuning_B3= model_B3.fit(
    train_ds,
    validation_data=val_ds, #this also creates val_accuracy data
    epochs=EPOCHS,
    callbacks=callbacks_list
)
#history is kinda like a black box or just all info, note it has history.history attritbute which we use to access everything.

43941136/43941136 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb3 (Functional)     │ (None, 1536)           │    10,783,535 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1536)           │         6,144 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │       786,944 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 202)            │        51,914 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,759,865 (44.86 MB)

 Trainable params: 11,669,490 (44.52 MB)

...

   1777/Unknown 372s 144ms/step - accuracy: 0.0073 - loss: 5.5994

2026-03-30 17:23:17.101125: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-30 17:23:17.293949: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-30 17:23:17.728900: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-30 17:23:17.945706: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


   1778/Unknown 428s 175ms/step - accuracy: 0.0073 - loss: 5.5993

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:160: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


1778/1778 ━━━━━━━━━━━━━━━━━━━━ 464s 196ms/step - accuracy: 0.0073 - loss: 5.5993 - val_accuracy: 0.0136 - val_loss: 5.2192 - learning_rate: 1.0000e-05
Epoch 2/30
1778/1778 ━━━━━━━━━━━━━━━━━━━━ 273s 153ms/step - accuracy: 0.0133 - loss: 5.3090 - val_accuracy: 0.0376 - val_loss: 5.0197 - learning_rate: 1.0000e-05
Epoch 3/30
1778/1778 ━━━━━━━━━━━━━━━━━━━━ 272s 152ms/step - accuracy: 0.0275 - loss: 5.1071 - val_accuracy: 0.0771 - val_loss: 4.7747 - learning_rate: 1.0000e-05
Epoch 4/30
1778/1778 ━━━━━━━━━━━━━━━━━━━━ 271s 152ms/step - accuracy: 0.0514 - loss: 4.8751 - val_accuracy: 0.1236 - val_loss: 4.4797 - learning_rate: 1.0000e-05
Epoch 5/30
1778/1778 ━━━━━━━━━━━━━━━━━━━━ 272s 153ms/step - accuracy: 0.0893 - loss: 4.6182 - val_accuracy: 0.1709 - val_loss: 4.1681 - learning_rate: 1.0000e-05
Epoch 6/30
1778/1778 ━━━━━━━━━━━━━━━━━━━━ 272s 153ms/step - accuracy: 0.1295 - loss: 4.3562 - val_accuracy: 0.2218 - val_loss: 3.8689 - learning_rate: 1.0000e-05
Epoch 7/30
1778/1778 ━━━━━━━━━━━━━━━━━━

In [13]:
history_df = pd.read_csv('/kaggle/working/training_log_B3_fixed.csv')

# Plotting the Loss (the most important part for BirdCLEF)
plt.figure(figsize=(10, 6))
plt.plot(history_df['epoch'], history_df['loss'], label='Training Loss')
plt.plot(history_df['epoch'], history_df['val_loss'], label='Validation Loss')
plt.title('B3 Fine-Tuning Progress (from CSV)')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/training_log_B3_fixed.csv'

In [ ]:
import tensorflow as tf
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

In [ ]:
!ls -l /kaggle/working/

In [ ]:
train_files = os.listdir('/kaggle/working/train_tfrecords')
val_files = os.listdir('/kaggle/working/val_tfrecords')

print(f"Train shards found: {len(train_files)}")
print(f"Validation shards found: {len(val_files)}")

In [ ]:
!du -h /kaggle/working/train_tfrecords/*.tfrecord

In [ ]:
size = os.path.getsize('/kaggle/working/train_tfrecords/shard_0.tfrecord')
print(f"Shard 0 size: {size / (1024*1024):.2f} MB")

In [ ]:
row = train_df.iloc[0]

try:
    print(f"Testing path: {row['full_path']}")
    # Manual test of the function
    image, label = process_audio_efficientB3(tf.constant(row['full_path']), tf.constant(row['label_id']))
    print(f"Success! Image shape: {image.shape}")
except Exception as e:
    print(f"!!! FUNCTION FAILED !!!")
    print(f"Error Message: {e}")

In [ ]:
os.path.exists(train_df.iloc[0]['full_path'])

In [ ]:
print(cv2.__version__)

In [ ]:
print(f"Max Label ID: {df_filtered['label_id'].max()}")
print(f"Number of Unique IDs: {df_filtered['label_id'].nunique()}")

In [ ]:
for images, labels in train_ds.take(1):
    # 2. Check the raw numbers
    print(f"Max pixel value: {tf.reduce_max(images).numpy()}")
    print(f"Min pixel value: {tf.reduce_min(images).numpy()}")
    print(f"Mean pixel value: {tf.reduce_mean(images).numpy()}")
    print(f"Shape of one batch: {images.shape}")
    
    # 3. Visual check (Optional)
    import matplotlib.pyplot as plt
    plt.imshow(images[0].numpy().astype('uint8')) # uint8 expects 0-255
    plt.title(f"Label: {labels[0].numpy()}")
    plt.show()